In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.ticker import PercentFormatter
from collections import defaultdict, Counter
from scipy.optimize import linprog
from scipy.spatial import ConvexHull
import plotly.graph_objects as go

from mabc.src.rg_solver import ResourceGatheringVI

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Computer Modern Roman', 'serif'],
    'mathtext.fontset': 'cm',
    
    'font.size': 18,             # Base font size (up from 14)
    'axes.labelsize': 20,        # X and Y axis labels (up from 16)
    'axes.titlesize': 18,        # Individual Subplot titles (up from 14)
    'figure.titlesize': 22,      # Main Figure title/suptitle (up from 18)
    'legend.fontsize': 14,       # Legend text (up from 11)
    'xtick.labelsize': 16,       # X-axis numbers (up from 13)
    'ytick.labelsize': 16,       # Y-axis numbers (up from 13)
    
    'pdf.fonttype': 42,          # Ensures text is editable in PDF
    'ps.fonttype': 42
})

# Core RL & 4D Policy Utilities


In [ ]:
def collect_trajectories(solver, policy, episodes, max_steps=100):
    """Rolls out the policy using the True Stochastic P_tensor."""
    dataset = []
    for _ in range(episodes):
        s_idx = np.random.choice(solver.n_states, p=solver.rho)
        state = solver.states[s_idx]
        
        step_count = 0
        while state != solver.TERMINAL_STATE and step_count < max_steps:
            if state not in policy: break
            
            action = policy[state]
            dataset.append((state, action))
            
            p_transitions = solver.P_tensor[solver.state_to_idx[state], action, :]
            next_s_idx = np.random.choice(solver.n_states, p=p_transitions)
            state = solver.states[next_s_idx]
            step_count += 1   
    return dataset

def build_stochastic_policy_4d(solver, dataset):
    """Safely builds an empirical MLE policy covering the full 4D space."""
    counts = defaultdict(Counter)
    for s, a in dataset:
        counts[s][a] += 1
        
    policy = {}
    for state in solver.states:
        if state == solver.TERMINAL_STATE: continue
        if state[0] == solver.START_STATE[0] and state[1] == solver.START_STATE[1] and (state[2] == 1 or state[3] == 1):
            continue
            
        if state in counts:
            total = sum(counts[state].values())
            policy[state] = {a: count / total for a, count in counts[state].items()}
        else:
            policy[state] = {0: 0.25, 1: 0.25, 2: 0.25, 3: 0.25}
    return policy

def exact_policy_evaluation_4d(solver, policy_probs):
    """Analytically evaluates expected 3D returns via Bellman equations."""
    R_pi = np.zeros((solver.n_states, 3))
    P_pi = np.zeros((solver.n_states, solver.n_states))
    
    for i, state in enumerate(solver.states):
        if state == solver.TERMINAL_STATE:
            P_pi[i, i] = 1.0 
            continue
            
        probs = policy_probs.get(state, {0:0.25, 1:0.25, 2:0.25, 3:0.25})
        for a, prob in probs.items():
            if prob <= 0: continue
            R_pi[i] += prob * (solver.P_tensor[i, a, :] @ solver.R_tensor[i, a, :, :])
            P_pi[i, :] += prob * solver.P_tensor[i, a, :]
            
    A = np.eye(solver.n_states) - solver.gamma * P_pi
    V = np.linalg.solve(A, R_pi)
    return solver.rho @ V

# Pareto Geometry & Exact LP Suboptimality


In [ ]:
def get_strictly_pareto_front(returns, policies):
    """Filters out any strictly dominated policies from the OLS results."""
    strictly_pareto_returns = []
    strictly_pareto_policies = []
    
    for i, r_eval in enumerate(returns):
        is_dominated = False
        for j, r_ref in enumerate(returns):
            if i == j: continue
            if np.all(r_ref >= r_eval) and np.any(r_ref > r_eval):
                is_dominated = True
                break
        
        if not is_dominated:
            strictly_pareto_returns.append(r_eval)
            strictly_pareto_policies.append(policies[i])
            
    return np.array(strictly_pareto_returns), strictly_pareto_policies

def calc_exact_3d_linf_suboptimality(ret, pareto_points, obj_ranges):
    """Linear Programming Chebyshev L-infinity projection onto the 3D Pareto Hull."""
    N = len(pareto_points)
    
    c = np.zeros(N + 1)
    c[0] = -1.0  
    
    A_eq = np.zeros((1, N + 1))
    A_eq[0, 1:] = 1.0
    b_eq = np.array([1.0])
    
    A_ub = np.zeros((3, N + 1))
    b_ub = np.zeros(3)
    
    for i in range(3):
        A_ub[i, 0] = obj_ranges[i]
        A_ub[i, 1:] = -pareto_points[:, i]
        b_ub[i] = -ret[i]
        
    bounds = [(None, None)] + [(0, 1) for _ in range(N)]
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    
    return max(0.0, -res.fun) if res.success else 0.0

# Visualization Tools


In [ ]:
def visualize_pareto_front_3d(sorted_pareto_points):
    """Interactive Plotly 3D visualization of the continuous frontier."""
    x, y, z = sorted_pareto_points[:, 0], sorted_pareto_points[:, 1], sorted_pareto_points[:, 2]
    indices = [f"Policy Index: {i}" for i in range(len(sorted_pareto_points))]

    fig = go.Figure()
    fig.add_trace(go.Mesh3d(
        x=x, y=y, z=z, alphahull=0, color='cyan', opacity=0.3, flatshading=True,
        name="Pareto Surface", hoverinfo="skip"
    ))
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z, mode='markers', text=indices,
        marker=dict(size=6, color=z, colorscale='Viridis', opacity=0.9, line=dict(color='white', width=1)),
        hovertemplate="<b>%{text}</b><br>Penalty: %{x:.4f}<br>Gold: %{y:.4f}<br>Gem: %{z:.4f}<extra></extra>",
        name="Experts"
    ))
    fig.update_layout(
        title=dict(text=f"3D Pareto Front (N={len(sorted_pareto_points)})", x=0.5, font=dict(size=20, color='white')),
        template="plotly_dark",
        scene=dict(xaxis_title="Enemy Penalty", yaxis_title="Gold Collected", zaxis_title="Gem Collected", aspectmode='cube'),
        margin=dict(l=0, r=0, b=0, t=50)
    )
    fig.show()

def draw_rg_grid_slice(ax, policy, title="", highlight_conflicts=False):
    """Projects a 4D policy onto a 2D map (Assumes 0 inventory)."""
    HOME = (4, 2)              # Bottom center
    GOLD_POS = (0, 2)          # Top center
    GEM_POS = (1, 4)           # Top right, one row down
    ENEMIES = [(0, 3), (1, 2)] # The two classic dragons

    action_map = {0: (0, 0.4), 1: (0, -0.4), 2: (-0.4, 0), 3: (0.4, 0)}
    ax.set_facecolor('#e8f4f8')
    
    for r in range(5):
        for c in range(5):
            state_slice = (r, c, 0, 0)
            y_pos = 4 - r
            color, label, is_terminal = "white", "", False
            
            if (r, c) == HOME: color, label, is_terminal = "lightblue", "H", True
            elif (r, c) == GOLD_POS: color, label = "gold", "Au"
            elif (r, c) == GEM_POS: color, label = "cyan", "Gem"
            elif (r, c) in ENEMIES: color, label, is_terminal = "salmon", "E", True
                
            is_random, is_conflict = False, False
            if not is_terminal and state_slice in policy:
                probs = policy[state_slice]
                if isinstance(probs, dict):
                    if len(probs) == 4 and all(p == 0.25 for p in probs.values()):
                        is_random, color = True, "#ffcccc"
                    elif len(probs) > 1:
                        is_conflict = True
                        color = "#ffe6e6" if highlight_conflicts else "#e6ffe6"
            
            ax.add_patch(patches.Rectangle((c - 0.5, y_pos - 0.5), 1, 1, linewidth=1, edgecolor='gray', facecolor=color, alpha=0.8))
            if label: ax.text(c, y_pos, label, ha='center', va='center', fontweight='bold', zorder=10)
                
            if not is_terminal and state_slice in policy:
                actions = policy[state_slice] if isinstance(policy[state_slice], dict) else {policy[state_slice]: 1.0}
                for a, prob in actions.items():
                    if prob == 0: continue
                    dx, dy = action_map[a]
                    aw, hw, alpha = 0.02 + (0.08 * prob), 0.15 + (0.15 * prob), 0.3 + (0.7 * prob)
                    arr_color = 'red' if is_random else ('blue' if is_conflict else 'black')
                    ax.arrow(c - (dx*0.1), y_pos - (dy*0.1), dx*0.8, dy*0.8, width=aw, head_width=hw, head_length=hw, 
                             fc=arr_color, ec=arr_color, alpha=alpha, length_includes_head=True)

    ax.set_xlim(-0.5, 4.5); ax.set_ylim(-0.5, 4.5)
    ax.set_xticks([]); ax.set_yticks([])
    if title: ax.set_title(title, fontweight='bold')

def plot_learning_curve(trajectory_steps, results, filename="rg_learning_curve.pdf"):
    """Plots the exact L_inf suboptimality with strict zero-bound clipping."""
    fig = plt.figure(figsize=(5, 5.2))
    colors = {"mixed": "red", "iso": "orange", "aug": "green"}
    labels = {"mixed": "Naive BC (Joint Dataset)", 
              "iso": "Isolated BC (Independent Experts)", 
              "aug": "MA-BC (Ours)"}
    markers = {"mixed": "d", "iso": "o", "aug": "s"}

    for k in ["mixed", "iso", "aug"]:
        means = np.array(results[k]["mean"])
        stds = np.array(results[k]["std"])/np.sqrt(100) 
        plt.plot(trajectory_steps, means, marker=markers[k], color=colors[k], linewidth=2.5, label=labels[k])
        plt.fill_between(trajectory_steps, np.maximum(0, means - stds), means + stds, color=colors[k], alpha=0.15, edgecolor='none')
    fig.subplots_adjust(top=0.90, bottom=0.25, left=0.24, right=0.98)
    plt.xlabel(r"Dataset Size per Expert")
    plt.ylabel(r"Normalized $L_\infty$ Distance to PF")
    plt.xticks(trajectory_steps)
    plt.gca().yaxis.set_major_formatter(PercentFormatter(1.0))
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(loc="upper right")
    plt.ylim(bottom=0)
    plt.tight_layout()
    plt.savefig(filename, transparent=True, dpi=300)
    plt.show()

# Suboptimality Experiment Loop


In [ ]:
def run_rg_scaling_experiment(solver, pol_A, pol_B, pareto_points, traj_steps, num_trials=10):
    """Executes the core dataset splitting, imitation learning, and LP evaluation."""
    obj_ranges = np.array([1.0, 1.0, 2.0]) 
    
    results = {"mixed": {"mean": [], "std": []}, "iso": {"mean": [], "std": []}, "aug": {"mean": [], "std": []}}
    
    for n_traj in traj_steps:
        m_gaps, i_gaps, a_gaps = [], [], []
        
        for _ in range(num_trials):
            data_A = collect_trajectories(solver, pol_A, n_traj)
            data_B = collect_trajectories(solver, pol_B, n_traj)
            
            vA, vB = dict(data_A), dict(data_B)
            shared_B = [(s, a) for (s, a) in data_B if s not in vA or vA[s] == a]
            shared_A = [(s, a) for (s, a) in data_A if s not in vB or vB[s] == a]

            pi_mix = build_stochastic_policy_4d(solver, data_A + data_B)
            pi_iso_A, pi_iso_B = build_stochastic_policy_4d(solver, data_A), build_stochastic_policy_4d(solver, data_B)
            pi_aug_A, pi_aug_B = build_stochastic_policy_4d(solver, data_A + shared_B), build_stochastic_policy_4d(solver, data_B + shared_A)
            
            ret_mix = exact_policy_evaluation_4d(solver, pi_mix)
            ret_iso_A = exact_policy_evaluation_4d(solver, pi_iso_A)
            ret_iso_B = exact_policy_evaluation_4d(solver, pi_iso_B)
            ret_aug_A = exact_policy_evaluation_4d(solver, pi_aug_A)
            ret_aug_B = exact_policy_evaluation_4d(solver, pi_aug_B)
            
            gap_mix = calc_exact_3d_linf_suboptimality(ret_mix, pareto_points, obj_ranges)
            
            gap_iso_A = calc_exact_3d_linf_suboptimality(ret_iso_A, pareto_points, obj_ranges)
            gap_iso_B = calc_exact_3d_linf_suboptimality(ret_iso_B, pareto_points, obj_ranges)
            gap_iso = (gap_iso_A + gap_iso_B) / 2.0
            
            gap_aug_A = calc_exact_3d_linf_suboptimality(ret_aug_A, pareto_points, obj_ranges)
            gap_aug_B = calc_exact_3d_linf_suboptimality(ret_aug_B, pareto_points, obj_ranges)
            gap_aug = (gap_aug_A + gap_aug_B) / 2.0
            
            m_gaps.append(gap_mix)
            i_gaps.append(gap_iso)
            a_gaps.append(gap_aug)
            
        results["mixed"]["mean"].append(np.mean(m_gaps)); results["mixed"]["std"].append(np.std(m_gaps))
        results["iso"]["mean"].append(np.mean(i_gaps)); results["iso"]["std"].append(np.std(i_gaps))
        results["aug"]["mean"].append(np.mean(a_gaps)); results["aug"]["std"].append(np.std(a_gaps))
        
        print(f"Traj: {n_traj:3d} | Mixed: {results['mixed']['mean'][-1]:.1%} | Iso: {results['iso']['mean'][-1]:.1%} | Aug: {results['aug']['mean'][-1]:.1%}")
        
    return results

# Execution: Find Pareto Front & Visualize Geometry

In [ ]:
solver = ResourceGatheringVI(gamma=0.7, init_state_dist="uniform")
ccs_results = solver.find_ccs_ols()

raw_points = np.array([r for r, p in ccs_results])
raw_policies = [p for r, p in ccs_results]

sorted_pareto_points, sorted_pareto_policies = get_strictly_pareto_front(raw_points, raw_policies)

visualize_pareto_front_3d(sorted_pareto_points)

# Execution: Grid Space Policy Divergence

In [ ]:
gold_idx = np.argmax(sorted_pareto_points[:, 1])  # Maximize Gold
gem_idx = np.argmax(sorted_pareto_points[:, 2])   # Maximize Gem

print(f"Max Gold Expert Return: {sorted_pareto_points[gold_idx]}")
print(f"Max Gem Expert Return:  {sorted_pareto_points[gem_idx]}")

pol_A_true = sorted_pareto_policies[gold_idx]
pol_B_true = sorted_pareto_policies[gem_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
fig.patch.set_facecolor('#f8f9fa')

draw_rg_grid_slice(axes[0], pol_A_true, title="Expert A (Max Gold)")
draw_rg_grid_slice(axes[1], pol_B_true, title="Expert B (Max Gem)")
plt.tight_layout()
plt.show()

# Execution: Evaluate MA-BC Sample Efficiency

In [ ]:
trajectory_steps = np.linspace(0, 200, 10).astype(int) 

results = run_rg_scaling_experiment(
    solver=solver, 
    pol_A=pol_A_true, 
    pol_B=pol_B_true, 
    pareto_points=sorted_pareto_points, 
    traj_steps=trajectory_steps, 
    num_trials=100
)

plot_learning_curve(trajectory_steps, results)

In [ ]:

def plot_rg_alpha_sweep_fixed_N(alphas=np.linspace(0.0, 1.0, 11),
                                fixed_n_traj=40, 
                                num_trials=50):
    print("="*65)
    print(f" EVALUATING RG ALPHA SWEEP AT FIXED N = {fixed_n_traj} ")
    print("="*65 + "\n")
    
    iso_means, iso_sems = [], []
    aug_means, aug_sems = [], []
    
    obj_ranges = np.array([1.0, 1.0, 2.0]) 

    for alpha in alphas:
        print(f"--- Processing Alpha = {alpha:.2f} ---")
        
        solver = ResourceGatheringVI(gamma=0.7, init_state_dist=alpha)
        
        ccs_results = solver.find_ccs_ols()
        raw_points = np.array([r for r, p in ccs_results])
        raw_policies = [p for r, p in ccs_results]
        
        pareto_points, pareto_policies = get_strictly_pareto_front(raw_points, raw_policies)
        
        gold_idx = np.argmax(pareto_points[:, 1])  # Maximize Gold
        gem_idx = np.argmax(pareto_points[:, 2])   # Maximize Gem
        pol_A_true = pareto_policies[gold_idx]
        pol_B_true = pareto_policies[gem_idx]

        gaps_iso, gaps_aug = [], []
        
        for _ in range(num_trials):
            data_A = collect_trajectories(solver, pol_A_true, fixed_n_traj)
            data_B = collect_trajectories(solver, pol_B_true, fixed_n_traj)
            
            vA, vB = dict(data_A), dict(data_B)
            shared_B = [(s, a) for (s, a) in data_B if s not in vA or vA[s] == a]
            shared_A = [(s, a) for (s, a) in data_A if s not in vB or vB[s] == a]

            pi_iso_A = build_stochastic_policy_4d(solver, data_A)
            pi_iso_B = build_stochastic_policy_4d(solver, data_B)
            pi_aug_A = build_stochastic_policy_4d(solver, data_A + shared_B)
            pi_aug_B = build_stochastic_policy_4d(solver, data_B + shared_A)

            ret_iso_A = exact_policy_evaluation_4d(solver, pi_iso_A)
            ret_iso_B = exact_policy_evaluation_4d(solver, pi_iso_B)
            ret_aug_A = exact_policy_evaluation_4d(solver, pi_aug_A)
            ret_aug_B = exact_policy_evaluation_4d(solver, pi_aug_B)

            gap_iso_A = calc_exact_3d_linf_suboptimality(ret_iso_A, pareto_points, obj_ranges)
            gap_iso_B = calc_exact_3d_linf_suboptimality(ret_iso_B, pareto_points, obj_ranges)
            gaps_iso.append((gap_iso_A + gap_iso_B) / 2.0)
            
            gap_aug_A = calc_exact_3d_linf_suboptimality(ret_aug_A, pareto_points, obj_ranges)
            gap_aug_B = calc_exact_3d_linf_suboptimality(ret_aug_B, pareto_points, obj_ranges)
            gaps_aug.append((gap_aug_A + gap_aug_B) / 2.0)
            
        iso_means.append(np.mean(gaps_iso))
        iso_sems.append(np.std(gaps_iso) / np.sqrt(num_trials))
        aug_means.append(np.mean(gaps_aug))
        aug_sems.append(np.std(gaps_aug) / np.sqrt(num_trials))

    iso_means, iso_sems = np.array(iso_means), np.array(iso_sems)
    aug_means, aug_sems = np.array(aug_means), np.array(aug_sems)

    fig = plt.figure(figsize=(5, 5.2))
    ax = fig.gca()
    ax.set_facecolor('#f8f9fa')
    
    ax.plot(alphas, iso_means, 'o-', color='orange', linewidth=3, markersize=8, label='Isolated BC')
    ax.fill_between(alphas, np.maximum(0, iso_means - iso_sems), iso_means + iso_sems, color='orange', alpha=0.15)
    
    ax.plot(alphas, aug_means, 's-', color='green', linewidth=3, markersize=8, label='MA-BC (Ours)')
    ax.fill_between(alphas, np.maximum(0, aug_means - aug_sems), aug_means + aug_sems, color='green', alpha=0.2)

    ax.set_title(fr"Resource Gathering | $N_1=N_2={fixed_n_traj}$")
    ax.set_xlabel(r"Probability of Random Spawn")
    ax.set_ylabel(r"Normalized $L_\infty$ Distance to PF")
    
    ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    
    ax.set_xticks(np.linspace(0.0, 1.0, 6))
    
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.set_ylim(bottom=0)
    
    fig.subplots_adjust(top=0.90, bottom=0.25, left=0.24, right=0.98)
    
    plt.savefig("rg_alpha_sweep.pdf", bbox_inches=None, dpi=300, transparent=True)
    plt.show()

alpha_values = np.linspace(0.0, 1.0, 11)
plot_rg_alpha_sweep_fixed_N(alphas=alpha_values, fixed_n_traj=80, num_trials=100)